# Phase 5: Risk-Aware Decision Policies

## Goals
- Implement tiered decision policies using calibrated probabilities
- Compare business metrics: uncalibrated vs calibrated decisions
- Optimize thresholds to minimize total expected cost
- Visualize Pareto frontiers of fraud loss vs operational cost
- Demonstrate the practical value of calibration for deployment

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.helpers import load_config, set_seed
from src.data.loader import load_creditcard, preprocess_creditcard
from src.data.splits import temporal_split_creditcard
from src.models.baselines import get_model
from src.calibration.post_hoc import get_calibrator
from src.evaluation.business_metrics import compute_business_metrics, pareto_frontier
from src.evaluation.visualization import plot_decision_distribution, plot_pareto_frontier
from src.policies.decision import (
    TieredDecisionPolicy,
    AdaptiveThresholdOptimizer,
    compare_calibrated_vs_uncalibrated_policies,
)

config = load_config('../configs/config.yaml')
set_seed(config['experiment']['seed'])
print('Setup complete.')

In [ ]:
# Load data, train model, calibrate
df_raw = load_creditcard(f"../{config['data']['creditcard_path']}")
X, y = preprocess_creditcard(df_raw)
splits = temporal_split_creditcard(df_raw)

X_train, y_train = X.iloc[splits['train']].values, y[splits['train']]
X_cal, y_cal = X.iloc[splits['calibration']].values, y[splits['calibration']]
X_test, y_test = X.iloc[splits['test']].values, y[splits['test']]

model = get_model('xgboost', config['models']['xgboost'])
model.fit(X_train, y_train)

raw_probs = model.predict_proba(X_test)
raw_probs_cal = model.predict_proba(X_cal)

# Best calibrator from Phase 3 (update based on your results)
best_calibrator = get_calibrator('isotonic_regression')
best_calibrator.fit(raw_probs_cal, y_cal)
cal_probs = best_calibrator.calibrate(raw_probs)

print(f'Test set: {len(y_test)} samples, {y_test.sum()} fraud')

## Decision Distribution Comparison

In [ ]:
fig = plot_decision_distribution(
    raw_probs, y_test,
    thresholds=config['evaluation']['decision_thresholds'],
    title='Uncalibrated Decision Distribution'
)
plt.savefig('../results/figures/decision_uncalibrated.png', dpi=150, bbox_inches='tight')
plt.show()

fig = plot_decision_distribution(
    cal_probs, y_test,
    thresholds=config['evaluation']['decision_thresholds'],
    title='Calibrated Decision Distribution'
)
plt.savefig('../results/figures/decision_calibrated.png', dpi=150, bbox_inches='tight')
plt.show()

## Business Metrics: Uncalibrated vs Calibrated

In [ ]:
uncal_biz = compute_business_metrics(
    raw_probs, y_test,
    thresholds=config['evaluation']['decision_thresholds'],
    costs=config['evaluation']['costs']
)

cal_biz = compute_business_metrics(
    cal_probs, y_test,
    thresholds=config['evaluation']['decision_thresholds'],
    costs=config['evaluation']['costs']
)

comparison = pd.DataFrame({'Uncalibrated': uncal_biz, 'Calibrated': cal_biz}).T
print(comparison.to_markdown())
comparison.to_csv('../results/tables/business_metrics_comparison.csv')
comparison

## Threshold Optimization

In [ ]:
# Optimize thresholds for calibrated probabilities
optimizer = AdaptiveThresholdOptimizer(costs=config['evaluation']['costs'])

print('Optimizing thresholds for UNCALIBRATED probs...')
uncal_opt = optimizer.optimize(raw_probs, y_test)
print(f'  Best thresholds: {uncal_opt["best_thresholds"]}')
print(f'  Best total cost: ${uncal_opt["best_total_cost"]:.2f}')

print('\nOptimizing thresholds for CALIBRATED probs...')
cal_opt = optimizer.optimize(cal_probs, y_test)
print(f'  Best thresholds: {cal_opt["best_thresholds"]}')
print(f'  Best total cost: ${cal_opt["best_total_cost"]:.2f}')

if uncal_opt['best_total_cost'] > 0:
    reduction = (uncal_opt['best_total_cost'] - cal_opt['best_total_cost']) / uncal_opt['best_total_cost'] * 100
    print(f'\nCost reduction from calibration: {reduction:.2f}%')

## Pareto Frontier: Fraud Loss vs Operational Cost

In [ ]:
# Pareto frontier for calibrated probs
pareto_data = pareto_frontier(
    cal_probs, y_test,
    costs=config['evaluation']['costs']
)

fig = plot_pareto_frontier(pareto_data, title='Pareto Frontier: Calibrated Probabilities')
plt.savefig('../results/figures/pareto_frontier.png', dpi=150, bbox_inches='tight')
plt.show()

## Full Head-to-Head Comparison

In [ ]:
h2h = compare_calibrated_vs_uncalibrated_policies(
    raw_probs, cal_probs, y_test,
    costs=config['evaluation']['costs']
)

print('Head-to-Head Policy Comparison')
print('=' * 50)
print(f'Uncalibrated optimal cost: ${h2h["uncalibrated"]["best_total_cost"]:.2f}')
print(f'Calibrated optimal cost:   ${h2h["calibrated"]["best_total_cost"]:.2f}')
print(f'Cost reduction:            ${h2h["cost_reduction"]:.2f} ({h2h["cost_reduction_pct"]:.1f}%)')

## Key Findings

1. How does calibration change the decision distribution?
2. What is the monetary benefit of using calibrated probabilities?
3. How do optimal thresholds differ between uncalibrated and calibrated models?
4. What does the Pareto frontier reveal about the fraud-loss vs cost trade-off?
5. How would you recommend deploying this system based on the results?